In [0]:

# Databricks notebook source
# MAGIC %md
# MAGIC # Bronze: CRA contribution limits and Bank of Canada policy rates
# MAGIC Lands the two reference datasets the platform depends on, with lineage columns on every row.
# MAGIC
# MAGIC | Table | Source | Feeds |
# MAGIC |---|---|---|
# MAGIC | `cra_limits_raw` | Annual TFSA dollar limits published by the CRA (TFSA Guide RC4466) | `silver.cra_limits_clean` → `gold.dim_tfsa_limits` |
# MAGIC | `boc_rates_raw` | Target for the overnight rate, series `V39079`, from the Bank of Canada Valet API | `silver.boc_rates_clean` → `gold.fct_boc_policy_rates` |
# MAGIC
# MAGIC If the Valet API can't be reached, the existing rates table is kept rather than overwritten with nothing.
 
# COMMAND ----------
 
from datetime import datetime, timezone
from decimal import Decimal
 
import requests
from pyspark.sql.types import (
    DateType, DecimalType, IntegerType, StringType, StructField, StructType, TimestampType,
)
 
SCHEMA = "personal_finance_bronze"
CRA_TABLE = f"{SCHEMA}.cra_limits_raw"
BOC_TABLE = f"{SCHEMA}.boc_rates_raw"
 
BOC_SERIES = "V39079"   # Target for the overnight rate
BOC_URL = f"https://www.bankofcanada.ca/valet/observations/{BOC_SERIES}/json?start_date=2009-01-01"
 
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
ingested_at = datetime.now(timezone.utc)
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 1. CRA TFSA dollar limits
 
# COMMAND ----------
 
TFSA_LIMITS = {
    2009: 5000, 2010: 5000, 2011: 5000, 2012: 5000,
    2013: 5500, 2014: 5500, 2015: 10000,
    2016: 5500, 2017: 5500, 2018: 5500,
    2019: 6000, 2020: 6000, 2021: 6000, 2022: 6000,
    2023: 6500, 2024: 7000, 2025: 7000,
}
 
cra_schema = StructType([
    StructField("tax_year", IntegerType(), False),
    StructField("tfsa_limit", DecimalType(10, 2), False),
    StructField("source", StringType(), False),
    StructField("ingested_at", TimestampType(), False),
])
 
cra_rows = [
    (year, Decimal(limit), "CRA TFSA Guide (RC4466)", ingested_at)
    for year, limit in sorted(TFSA_LIMITS.items())
]
 
(
    spark.createDataFrame(cra_rows, cra_schema)
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(CRA_TABLE)
)
print(f"Wrote {len(cra_rows)} rows to {CRA_TABLE}")
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 2. Bank of Canada policy rate
 
# COMMAND ----------
 
boc_schema = StructType([
    StructField("rate_date", DateType(), False),
    StructField("policy_rate", DecimalType(5, 2), False),
    StructField("series_id", StringType(), False),
    StructField("source", StringType(), False),
    StructField("ingested_at", TimestampType(), False),
])
 
try:
    response = requests.get(BOC_URL, timeout=30)
    response.raise_for_status()
    observations = response.json()["observations"]
 
    boc_rows = [
        (
            datetime.strptime(obs["d"], "%Y-%m-%d").date(),
            Decimal(obs[BOC_SERIES]["v"]),
            BOC_SERIES,
            "Bank of Canada Valet API",
            ingested_at,
        )
        for obs in observations
        if obs.get(BOC_SERIES, {}).get("v") not in (None, "")
    ]
    assert boc_rows, "Valet API returned no observations"
 
    (
        spark.createDataFrame(boc_rows, boc_schema)
        .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(BOC_TABLE)
    )
    print(f"Wrote {len(boc_rows):,} rows to {BOC_TABLE}")
 
except Exception as e:
    # Don't wipe good data because the API is down or outbound internet is blocked
    if spark.catalog.tableExists(BOC_TABLE):
        print(f"Could not refresh rates ({e}). Keeping the existing {BOC_TABLE}.")
    else:
        raise RuntimeError(f"Could not load Bank of Canada rates and no existing table to keep: {e}")
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## 3. Validation
 
# COMMAND ----------
 
cra = spark.table(CRA_TABLE)
assert cra.count() == len(TFSA_LIMITS), "CRA limits row count doesn't match the source list"
assert cra.filter("tfsa_limit <= 0").count() == 0, "Found a non-positive TFSA limit"
assert cra.selectExpr("min(tax_year)").first()[0] == 2009, "TFSA limits must start in 2009"
 
boc = spark.table(BOC_TABLE)
assert boc.count() > 0, "Bank of Canada rates table is empty"
assert boc.filter("policy_rate < 0").count() == 0, "Found a negative policy rate"
 
print("Bronze ingestion complete. All checks passed.")
 